In [ ]:
# =====================================
# IMPORTS
# =====================================

from src.dataset import get_dataloaders
from src.models.ViT import get_vit_small
from src.train import train_model

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from collections import Counter


# =====================================
# DEVICE
# =====================================

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")


# =====================================
# DATA
# =====================================

train_loader, val_loader = get_dataloaders(
    "../data/balanced/clean_balanced_train",
    "../data/balanced/clean_balanced_train"
)


# =====================================
# CLASS WEIGHT COMPUTATION
# =====================================

def compute_class_weights(dataset, device):

    labels = [label for _, label in dataset]

    counts = Counter(labels)

    num_classes = len(counts)
    total = len(labels)

    weights = []

    for i in range(num_classes):
        weights.append(total / (num_classes * counts[i]))

    weights = torch.tensor(weights, dtype=torch.float32).to(device)

    return weights


class_weights = compute_class_weights(train_loader.dataset, device)

print("Class Weights:", class_weights)


# =====================================
# FOCAL LOSS
# =====================================

class FocalLoss(nn.Module):

    def __init__(self, weight=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.weight = weight
        self.gamma = gamma

    def forward(self, inputs, targets):

        ce_loss = F.cross_entropy(
            inputs,
            targets,
            weight=self.weight,
            reduction="none"
        )

        pt = torch.exp(-ce_loss)

        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        return focal_loss.mean()


criterion = FocalLoss(weight=class_weights, gamma=2.0)


# =====================================
# MODEL
# =====================================

model = get_vit_small(num_classes=7).to(device)


# =====================================
# OPTIMIZER + SCHEDULER
# =====================================

optimizer = optim.AdamW(model.parameters(), lr=3e-4)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10
)


# =====================================
# TRAINING
# =====================================

train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    device,
    model_name="efficientnet_focal_weights"
)

/Users/bsama/Desktop/Github emotion recognition/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Class Weights: tensor([0.9959, 1.1946, 1.0459, 0.8602, 0.8885, 1.0823, 1.0094],
       device='mps:0')
<== Training Started ==>
Model: efficientnet_focal_weights
Timestamp: 20260305_151646
Device: mps

Epoch 1/10
Train Acc: 40.61%
Val Acc:   53.67%
✅ New best model saved

Epoch 2/10
Train Acc: 54.88%
Val Acc:   58.41%
✅ New best model saved

Epoch 3/10
Train Acc: 59.00%
Val Acc:   57.38%

Epoch 4/10
Train Acc: 62.86%
Val Acc:   64.03%
✅ New best model saved

Epoch 5/10
Train Acc: 65.77%
Val Acc:   66.60%
✅ New best model saved

Epoch 6/10
Train Acc: 69.09%
Val Acc:   71.87%
✅ New best model saved

Epoch 7/10
Train Acc: 72.59%
Val Acc:   75.18%
✅ New best model saved

Epoch 8/10
Train Acc: 76.41%
Val Acc:   78.67%
✅ New best model saved
